> **Legacy notebook.** `compare_denoise.py` was removed; its PCA/ICA variants live in `scripts/compare_methods.py`, which compares every method with the same metrics. `remove_ripple_pca` now returns the input unchanged when auto-detection finds no ripple PCs. Cells importing `compare_denoise` will not run.

# PMT Ripple Denoising — Method Comparison

Compare three approaches to removing PMT ripple noise from 2P calcium imaging data:
1. **PCA subtraction** — remove leading PCs (fast, may mix in some signal)
2. **PCA + ICA rotation** — rotate PCA subspace with ICA to isolate ripple more cleanly
3. **Robust PCA** — L+S decomposition: low-rank ripple + sparse signal (slow)

Point `data_dir` at a session folder containing trial TIFFs.

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path

import sys

from bandaid.pca_clean import remove_ripple_pca, detect_ripple_pcs

In [ ]:
# ---- CONFIG ----
data_dir = Path("./data/HK13/HK13_20260814")

n_noise_pcs = 2           # PCs to remove (PCA and PCA+ICA methods)
rpca_lambda = None         # RPCA lambda (None = auto: 1/sqrt(max(T, pixels)))
max_frames_compare = 500   # max frames for PCA/ICA comparison (None = all)
max_frames_rpca = 200      # max frames for RPCA (it's slow — SVD every iteration)

## Load session trials

In [ ]:
trial_paths = sorted(data_dir.glob("*.tif")) + sorted(data_dir.glob("*.tiff"))
# trial_paths = trial_paths[2:6] # Select some trials
assert trial_paths, f"No tiff files found in {data_dir}"
print(f"Found {len(trial_paths)} trial files:\n")

trials = []
for p in trial_paths:
    data = tifffile.imread(str(p))
    if data.ndim == 2:
        data = data[np.newaxis]
    trials.append(data[::3])
    print(f"  {p.name}: {data.shape} {data.dtype}")

stack = np.concatenate(trials, axis=0)
trial_boundaries = np.cumsum([0] + [t.shape[0] for t in trials])  # for splitting later
print(f"\nFull stack: {stack.shape}  dtype: {stack.dtype}  ({stack.nbytes / 1e9:.1f} GB)")

In [ ]:
# Subsample frames for comparison (uniform across session)
if max_frames_compare is not None and stack.shape[0] > max_frames_compare:
    idx = np.linspace(0, stack.shape[0] - 1, max_frames_compare, dtype=int)
    stack_sub = stack[idx]
    print(f"Subsampled {stack.shape[0]} -> {stack_sub.shape[0]} frames for comparison")
else:
    stack_sub = stack
    print(f"Using all {stack_sub.shape[0]} frames")

## Quick look at the data

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

sample_frames = [0, len(stack_sub) // 3, 2 * len(stack_sub) // 3]
for i, fi in enumerate(sample_frames):
    vmin, vmax = np.percentile(stack_sub[fi], [2, 98])
    axes[i].imshow(stack_sub[fi], cmap="gray", vmin=vmin, vmax=vmax)
    axes[i].set_title(f"Frame {fi}")
    axes[i].axis("off")

mean_img = stack_sub.mean(axis=0)
axes[3].imshow(mean_img, cmap="gray")
axes[3].set_title("Temporal mean")
axes[3].axis("off")
plt.suptitle("Raw data overview", fontsize=12)
plt.tight_layout()
plt.show()

# 2D power spectrum — ripple shows as bright peaks away from center
mag = np.abs(np.fft.fftshift(np.fft.fft2(mean_img)))
mag[mag.shape[0] // 2, mag.shape[1] // 2] = 0  # zero DC
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.log1p(mag), cmap="inferno")
ax.set_title("2D power spectrum (log) — look for ripple peaks")
ax.axis("off")
plt.tight_layout()
plt.show()

## Method 1 — PCA subtraction

In [ ]:
cleaned_pca, info_pca = remove_ripple_pca(
    stack_sub, n_noise_pcs=n_noise_pcs, return_info=True
)

# Show the first few PCA components — noise PCs get red borders
H, W = stack_sub.shape[1], stack_sub.shape[2]
n_show = min(6, len(info_pca["explained_variance_ratio"]))
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 3.5))
for i in range(n_show):
    comp = info_pca["components"][i].reshape(H, W)
    axes[i].imshow(comp, cmap="RdBu_r")
    label = f"PC{i+1} ({info_pca['explained_variance_ratio'][i]*100:.1f}%)"
    if i in info_pca["noise_pcs"]:
        label += " <- NOISE"
        for spine in axes[i].spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    axes[i].set_title(label, fontsize=9)
    axes[i].axis("off")
plt.suptitle("PCA components (red = removed)", fontsize=11)
plt.tight_layout()
plt.show()

## Method 2 — PCA + ICA rotation

In [ ]:
cleaned_ica = denoise_pca_ica(stack_sub, n_noise_pcs=n_noise_pcs)

## Method 3 — Robust PCA

RPCA runs full SVD on every ALM iteration, so it's much slower. Using a smaller frame subset.

In [ ]:
rpca_n = min(max_frames_rpca, stack_sub.shape[0])
rpca_sub = stack_sub[:rpca_n]
print(f"Running RPCA on {rpca_n} frames ...")
cleaned_rpca = denoise_rpca(rpca_sub, lam=rpca_lambda)

## Comparison

In [ ]:
# Pick a frame that's within the RPCA subset so all three are comparable
mid = rpca_n // 2
frame_raw = stack_sub[mid]
vmin, vmax = np.percentile(frame_raw, [2, 98])

results = [cleaned_pca[mid], cleaned_ica[mid], cleaned_rpca[mid]]
labels = ["PCA", "PCA + ICA", "Robust PCA"]

n = len(results) + 1
fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 9))

# Top row: cleaned frames
axes[0, 0].imshow(frame_raw, cmap="gray", vmin=vmin, vmax=vmax)
axes[0, 0].set_title("Original", fontweight="bold")
axes[0, 0].axis("off")
axes[1, 0].axis("off")

for j, (result, label) in enumerate(zip(results, labels)):
    axes[0, j + 1].imshow(result, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0, j + 1].set_title(label, fontweight="bold")
    axes[0, j + 1].axis("off")

    # Bottom row: what was removed
    diff = frame_raw.astype(float) - result.astype(float)
    lim = np.percentile(np.abs(diff), 99)
    axes[1, j + 1].imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[1, j + 1].set_title(f"Removed ({label})", fontsize=9)
    axes[1, j + 1].axis("off")

plt.suptitle(f"PMT ripple removal comparison — frame {mid}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 2D power spectra — ripple peaks should vanish after cleaning
images = [frame_raw] + results
titles = ["Original"] + labels

fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 4.5))
for j, (img, title) in enumerate(zip(images, titles)):
    mag = np.abs(np.fft.fftshift(np.fft.fft2(img.astype(float))))
    mag[mag.shape[0] // 2, mag.shape[1] // 2] = 0
    axes[j].imshow(np.log1p(mag), cmap="inferno")
    axes[j].set_title(title, fontsize=10)
    axes[j].axis("off")
plt.suptitle("2D power spectra — ripple peaks should disappear", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into a region to inspect fine detail preservation
# Adjust the crop coordinates to a region with visible cells
y0, y1 = 100, 250
x0, x1 = 100, 250

crops_raw = frame_raw[y0:y1, x0:x1]
crops = [r[y0:y1, x0:x1] for r in results]
vmin_c, vmax_c = np.percentile(crops_raw, [2, 98])

fig, axes = plt.subplots(1, len(crops) + 1, figsize=(5 * (len(crops) + 1), 5))
axes[0].imshow(crops_raw, cmap="gray", vmin=vmin_c, vmax=vmax_c)
axes[0].set_title("Original (crop)", fontweight="bold")
axes[0].axis("off")
for j, (crop, label) in enumerate(zip(crops, labels)):
    axes[j + 1].imshow(crop, cmap="gray", vmin=vmin_c, vmax=vmax_c)
    axes[j + 1].set_title(f"{label} (crop)", fontweight="bold")
    axes[j + 1].axis("off")
plt.suptitle(f"Detail crop [{y0}:{y1}, {x0}:{x1}]", fontsize=12)
plt.tight_layout()
plt.show()

## Save corrected trials

Pick the winning method, apply to the full stack, and save each trial separately.

In [ ]:
# ---- Pick method ----
# Uncomment your winner:
# denoise_fn = lambda s: denoise_pca(s, n_noise_pcs=n_noise_pcs)
# denoise_fn = lambda s: denoise_pca_ica(s, n_noise_pcs=n_noise_pcs)
# denoise_fn = lambda s: remove_ripple_pca(s, n_noise_pcs=n_noise_pcs)
denoise_fn = lambda s: denoise_rpca(s)

# ---- Apply to each trial and save ----
out_dir = data_dir / "cleaned"
out_dir.mkdir(exist_ok=True)
for p, trial in zip(trial_paths, trials):
    print(f"Cleaning {p.name} ...")
    cleaned = denoise_fn(trial)
    cleaned_concat = np.concatenate([trial, cleaned], axis=2)
    out_path = out_dir / p.name
    tifffile.imwrite(str(out_path), cleaned_concat)
    print(f"  -> {out_path}")